# Attention Is All You Need

This notebook reproduces the core experiment from the seminal paper "Attention Is All You Need," which introduced the Transformer architecture. We focus on Neural Machine Translation (English to German).

To ensure the notebook runs efficiently within Colab's resource limits and time constraints, several simplifications have been made, particularly regarding model size, dataset size, and training duration. The implementation leverages PyTorch's built-in `nn.Transformer` modules where possible, alongside custom components like positional encoding.


In [ ]:
# Install necessary libraries
!pip install -q torch torchtext transformers datasets sentencepiece sacrebleu matplotlib

import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

from datasets import load_dataset
from transformers import AutoTokenizer
import sacrebleu
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Configure device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
# --- Data Loading and Preprocessing ---

print("Loading WMT 2014 English-to-German dataset...")
# Load the dataset specified in the reproducibility spec
train_dataset_raw = load_dataset('wmt14', 'de-en', split='train')

# Subset strategy: Use the first 5000 samples
subset_size = 5000
train_dataset_raw = train_dataset_raw.select(range(min(subset_size, len(train_dataset_raw))))

# Create a small validation split from the training data for quick evaluation
# Using the last 500 samples of the subset for validation
val_size = min(500, len(train_dataset_raw) // 5)
val_dataset_raw = train_dataset_raw.select(range(len(train_dataset_raw) - val_size, len(train_dataset_raw)))
train_dataset_raw = train_dataset_raw.select(range(len(train_dataset_raw) - val_size))

print(f"Training data size: {len(train_dataset_raw)}")
print(f"Validation data size: {len(val_dataset_raw)}")

# Load a pre-trained tokenizer for English-German translation
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")

# Define special tokens based on the tokenizer. For opus-mt-en-de, </s> is 0, <pad> is 65000.
# We will use 0 as both BOS and EOS for simplification, as tokenizer.bos_token_id and tokenizer.eos_token_id are both 0.
# PAD_IDX is the actual pad token ID from the tokenizer.
PAD_IDX = tokenizer.pad_token_id  # Typically 65000
BOS_IDX = tokenizer.convert_tokens_to_ids('</s>') # Typically 0
EOS_IDX = tokenizer.convert_tokens_to_ids('</s>') # Typically 0

print(f"Effective BOS_IDX: {BOS_IDX}, Effective EOS_IDX: {EOS_IDX}, Effective PAD_IDX: {PAD_IDX}")

def tokenize_function(examples):
    # Tokenize source (en) and target (de) sentences
    # The tokenizer handles adding appropriate special tokens like </s> automatically for output in 'labels'
    model_inputs = tokenizer(examples['en'], max_length=128, truncation=True, return_attention_mask=False)
    labels = tokenizer(examples['de'], max_length=128, truncation=True, return_attention_mask=False)

    model_inputs['labels'] = labels['input_ids']
    return model_inputs


# Apply tokenization to the datasets
tokenized_train_dataset = train_dataset_raw.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset_raw.map(tokenize_function, batched=True)

# Remove original text columns and set format to PyTorch tensors
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["en", "de"])
tokenized_val_dataset = tokenized_val_dataset.remove_columns(["en", "de"])


class TranslationDataset(Dataset):
    def __init__(self, tokenized_dataset, bos_idx, eos_idx, pad_idx):
        self.dataset = tokenized_dataset
        self.BOS_IDX = bos_idx
        self.EOS_IDX = eos_idx
        self.PAD_IDX = pad_idx

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        src = item['input_ids']
        tgt_labels_original = item['labels'] # e.g., [w1, w2, w3, EOS_IDX]

        src_tensor = torch.tensor(src, dtype=torch.long)
        
        # For the decoder input (tgt_input), prepend BOS_IDX to the original target sequence (without its EOS_IDX)
        # For the decoder label (tgt_label), use the original target sequence (including its EOS_IDX)
        # This setup aligns the sequence lengths for the transformer decoder input and the loss calculation target.
        # Example:
        # Original target tokens: [w1, w2, w3, EOS]
        # Decoder input:        [BOS, w1, w2, w3]
        # Decoder label:        [w1, w2, w3, EOS]

        # Construct full target sequence including BOS at start and EOS at end
        # Since `tgt_labels_original` already ends with `EOS_IDX` (0), we just prepend `BOS_IDX` (0)
        # So `full_target_sequence` will be `[BOS_IDX, w1, w2, w3, EOS_IDX]`
        full_target_sequence = torch.cat([torch.tensor([self.BOS_IDX], dtype=torch.long),
                                          torch.tensor(tgt_labels_original, dtype=torch.long)])

        # Decoder input (shifted right, remove last token)
        tgt_tensor_input = full_target_sequence[:-1]
        # Decoder label for loss (remove first token)
        tgt_tensor_label = full_target_sequence[1:]

        return src_tensor, tgt_tensor_input, tgt_tensor_label


train_translation_dataset = TranslationDataset(tokenized_train_dataset, BOS_IDX, EOS_IDX, PAD_IDX)
val_translation_dataset = TranslationDataset(tokenized_val_dataset, BOS_IDX, EOS_IDX, PAD_IDX)

# Collate function for DataLoader
def collate_batch(batch):
    src_batch, tgt_input_batch, tgt_label_batch = [], [], []
    for src_sample, tgt_input_sample, tgt_label_sample in batch:
        src_batch.append(src_sample)
        tgt_input_batch.append(tgt_input_sample)
        tgt_label_batch.append(tgt_label_sample)

    # Pad sequences to the length of the longest sequence in the batch
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_input_batch = pad_sequence(tgt_input_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_label_batch = pad_sequence(tgt_label_batch, padding_value=PAD_IDX, batch_first=True)

    return src_batch, tgt_input_batch, tgt_label_batch


BATCH_SIZE = 32

train_dataloader = DataLoader(train_translation_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_dataloader = DataLoader(val_translation_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

# Determine vocabulary sizes
# For `Helsinki-NLP/opus-mt-en-de`: tokenizer.vocab_size is 65000. tokenizer.pad_token_id is 65000.
# This means valid token IDs are 0 to 64999. ID 65000 is used for padding.
# So our embedding layer needs to handle up to ID 65000. Thus, num_embeddings should be 65001.
SRC_VOCAB_SIZE = tokenizer.vocab_size + 1 # Accounts for tokens 0...64999 and PAD_IDX=65000
TGT_VOCAB_SIZE = tokenizer.vocab_size + 1 # Same for target

print(f"Source Vocabulary Size: {SRC_VOCAB_SIZE}")
print(f"Target Vocabulary Size: {TGT_VOCAB_SIZE}")


In [ ]:
# --- Model Architecture ---

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1) # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model) # (1, max_len, d_model) for broadcasting with (batch, seq_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        x = x + self.pe[:, :x.size(1)] # Add positional encoding to the input embeddings
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, ntoken_src: int, ntoken_tgt: int, d_model: int, nhead: int, d_hid: int,
                 nlayers: int, dropout: float = 0.5):
        super().__init__()
        self.model_type = 'Transformer'
        self.src_embedding = nn.Embedding(ntoken_src, d_model)
        self.tgt_embedding = nn.Embedding(ntoken_tgt, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        # PyTorch's nn.TransformerEncoderLayer and nn.TransformerDecoderLayer are used
        # batch_first=True makes inputs/outputs (batch_size, seq_len, feature_dim)
        transformer_encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, d_hid, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(transformer_encoder_layer, nlayers)

        transformer_decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, d_hid, dropout, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(transformer_decoder_layer, nlayers)

        self.generator = nn.Linear(d_model, ntoken_tgt)
        self.d_model = d_model

        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        self.src_embedding.weight.data.uniform_(-initrange, initrange)
        self.tgt_embedding.weight.data.uniform_(-initrange, initrange)
        self.generator.bias.data.zero_()
        self.generator.weight.data.uniform_(-initrange, initrange)

    def encode(self, src: torch.Tensor, src_padding_mask: torch.Tensor) -> torch.Tensor:
        # src: (batch_size, src_seq_len)
        src = self.src_embedding(src) * math.sqrt(self.d_model) # (batch_size, src_seq_len, d_model)
        src = self.pos_encoder(src) # (batch_size, src_seq_len, d_model)
        # src_mask is not used for encoder (full attention on source) but can be passed as None
        return self.transformer_encoder(src, src_key_padding_mask=src_padding_mask)

    def decode(self, tgt: torch.Tensor, memory: torch.Tensor, tgt_mask: torch.Tensor,
               tgt_padding_mask: torch.Tensor, memory_key_padding_mask: torch.Tensor) -> torch.Tensor:
        # tgt: (batch_size, tgt_seq_len)
        tgt = self.tgt_embedding(tgt) * math.sqrt(self.d_model) # (batch_size, tgt_seq_len, d_model)
        tgt = self.pos_encoder(tgt) # (batch_size, tgt_seq_len, d_model)
        return self.transformer_decoder(tgt, memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_padding_mask, memory_key_padding_mask=memory_key_padding_mask)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor, src_mask: torch.Tensor, tgt_mask: torch.Tensor, 
                src_padding_mask: torch.Tensor, tgt_padding_mask: torch.Tensor, memory_key_padding_mask: torch.Tensor):
        # src_mask is passed as None for encoder (full attention over source)
        memory = self.encode(src, src_padding_mask)
        output = self.decode(tgt, memory, tgt_mask, tgt_padding_mask, memory_key_padding_mask)
        return self.generator(output)


# Model Hyperparameters (simplified as per reproducibility spec)
D_MODEL = 128      # Embedding dimension
NHEAD = 4          # Number of attention heads
D_HID = 256        # Dimension of the feedforward network model in `nn.TransformerEncoderLayer`
NLAYERS = 2        # Number of encoder and decoder layers
DROPOUT = 0.1      # Dropout rate

model = TransformerModel(SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, D_MODEL, NHEAD, D_HID, NLAYERS, DROPOUT).to(device)

print("Transformer Model initialized with simplified parameters:")
print(f"  d_model: {D_MODEL}")
print(f"  nhead: {NHEAD}")
print(f"  d_hid: {D_HID}")
print(f"  nlayers: {NLAYERS}")
print(f"  Source Vocab Size: {SRC_VOCAB_SIZE}")
print(f"  Target Vocab Size: {TGT_VOCAB_SIZE}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")


In [ ]:
# --- Training Setup ---

LEARNING_RATE = 0.0001
EPOCHS = 2

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)
# The original paper uses a custom learning rate scheduler, but we use a fixed rate as per spec.

# CrossEntropyLoss ignores padding tokens in the target labels
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# --- Masking Functions ---

def generate_square_subsequent_mask(sz):
    # For target sequence, prevents attention to future tokens
    # Output shape: (sz, sz)
    mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask.to(device)

def create_padding_mask(seq, pad_idx):
    # For source and target padding, masks out padding tokens
    # Output shape: (batch_size, seq_len)
    return (seq == pad_idx).to(device)


# --- Training and Evaluation Functions ---

def train_epoch(model, dataloader, optimizer, criterion, PAD_IDX, device):
    model.train()
    losses = 0
    for src, tgt_input, tgt_label in tqdm(dataloader, desc="Training"):
        src, tgt_input, tgt_label = src.to(device), tgt_input.to(device), tgt_label.to(device)

        # Create masks
        src_padding_mask = create_padding_mask(src, PAD_IDX)
        tgt_padding_mask = create_padding_mask(tgt_input, PAD_IDX)
        tgt_mask = generate_square_subsequent_mask(tgt_input.size(1))
        
        # memory_key_padding_mask for TransformerDecoder masks the *encoder output* (memory)
        # It should be based on the source padding mask.
        memory_key_padding_mask = src_padding_mask

        optimizer.zero_grad()
        
        # Model input shapes are (batch_size, seq_len)
        # output shape: (batch_size, tgt_seq_len, ntoken_tgt)
        output = model(src, tgt_input, None, tgt_mask, src_padding_mask, tgt_padding_mask, memory_key_padding_mask)

        # Reshape for CrossEntropyLoss: (N, C) for input, (N) for target
        # We flatten the batch and sequence dimensions
        loss = criterion(output.reshape(-1, output.shape[-1]), tgt_label.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Gradient clipping as suggested in paper
        optimizer.step()
        losses += loss.item()
    return losses / len(dataloader)

def evaluate(model, dataloader, criterion, PAD_IDX, device):
    model.eval()
    losses = 0
    with torch.no_grad():
        for src, tgt_input, tgt_label in tqdm(dataloader, desc="Validation"):
            src, tgt_input, tgt_label = src.to(device), tgt_input.to(device), tgt_label.to(device)

            src_padding_mask = create_padding_mask(src, PAD_IDX)
            tgt_padding_mask = create_padding_mask(tgt_input, PAD_IDX)
            tgt_mask = generate_square_subsequent_mask(tgt_input.size(1))
            memory_key_padding_mask = src_padding_mask

            output = model(src, tgt_input, None, tgt_mask, src_padding_mask, tgt_padding_mask, memory_key_padding_mask)
            loss = criterion(output.reshape(-1, output.shape[-1]), tgt_label.reshape(-1))
            losses += loss.item()
    return losses / len(dataloader)


# --- Training Loop ---
print(f"Starting training for {EPOCHS} epochs...")
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    epoch_loss = train_epoch(model, train_dataloader, optimizer, criterion, PAD_IDX, device)
    val_loss = evaluate(model, val_dataloader, criterion, PAD_IDX, device)
    train_losses.append(epoch_loss)
    val_losses.append(val_loss)

    print(f"Epoch: {epoch}, Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_transformer_model.pt')
        print("  -> Saved best model state.")

print("Training complete.")


In [ ]:
# --- Evaluation ---

# Load the best model
model.load_state_dict(torch.load('best_transformer_model.pt'))
model.eval()

# Helper function for greedy decoding
def greedy_decode(model, src, src_padding_mask, max_len, start_symbol, end_symbol, device):
    src = src.to(device) # (1, src_seq_len)
    src_padding_mask = src_padding_mask.to(device) # (1, src_seq_len)

    memory = model.encode(src, src_padding_mask) # (1, src_seq_len, d_model)
    
    # Start with a batch of start symbols
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(device) # (1, 1)

    for i in range(max_len - 1):
        tgt_mask = generate_square_subsequent_mask(ys.size(1)) # (current_tgt_seq_len, current_tgt_seq_len)
        # No padding mask needed for single token generation at each step for tgt_padding_mask
        # memory_key_padding_mask comes from src_padding_mask
        # out: (1, current_tgt_seq_len, d_model)
        out = model.decode(ys, memory, tgt_mask, None, src_padding_mask)
        out = out[:, -1, :] # Take the last token's prediction (1, d_model)
        prob = model.generator(out) # (1, ntoken_tgt)
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys = torch.cat([ys, torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=1)
        if next_word == end_symbol:
            break
    return ys

def translate_sentence(model, sentence: str, tokenizer, BOS_IDX, EOS_IDX, PAD_IDX, device, max_len: int = 128):
    model.eval()
    # Tokenize source sentence
    src_tokens = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=max_len).input_ids.to(device)
    
    # Create source padding mask
    src_padding_mask = create_padding_mask(src_tokens, PAD_IDX)

    # Greedy decode
    translated_tokens = greedy_decode(model, src_tokens, src_padding_mask, max_len, BOS_IDX, EOS_IDX, device).squeeze(0)

    # Convert tensor of token IDs to list for decoding
    translated_tokens_list = translated_tokens.tolist()

    # Remove BOS/EOS tokens if they were explicitly added for decoding
    # Given BOS_IDX and EOS_IDX are 0 for this tokenizer, and it uses 0 as </s>
    # tokenizer.decode with skip_special_tokens=True will handle 0 and 65000 (PAD_IDX) correctly.
    # No manual removal of BOS/EOS from list needed, just filter PAD.
    filtered_tokens = [token_id for token_id in translated_tokens_list if token_id != PAD_IDX]
    translated_text = tokenizer.decode(filtered_tokens, skip_special_tokens=True)
    return translated_text


print("\n--- Sample Translations ---")
example_sentences_en = [
    "Hello, how are you?",
    "The quick brown fox jumps over the lazy dog.",
    "Germany is a country in Central Europe.",
    "Machine learning is a fascinating field."
]

for i, sentence in enumerate(example_sentences_en):
    translated_text = translate_sentence(model, sentence, tokenizer, BOS_IDX,
                                         EOS_IDX, PAD_IDX, device)
    print(f"English: {sentence}")
    print(f"German (translated): {translated_text}\n")


print("\n--- BLEU Score Calculation (on a small validation subset) ---")
alist_of_references = []
alist_of_hypotheses = []

# Using a small subset of the validation dataloader for quick BLEU calculation
num_samples_for_bleu = min(200, len(val_translation_dataset)) # Reduced further for speed

print(f"Calculating BLEU for {num_samples_for_bleu} validation samples...")

# Create a DataLoader for single-sentence processing for BLEU
bleu_dataset = val_translation_dataset.dataset.select(range(num_samples_for_bleu))
bleu_dataloader = DataLoader(TranslationDataset(bleu_dataset, BOS_IDX, EOS_IDX, PAD_IDX),
                                batch_size=1, shuffle=False, collate_fn=collate_batch)

model.eval()
with torch.no_grad():
    for i, (src, _, tgt_label) in tqdm(enumerate(bleu_dataloader), total=num_samples_for_bleu, desc="Generating translations for BLEU"):
        src = src.to(device) # (1, src_seq_len)
        src_padding_mask = create_padding_mask(src, PAD_IDX)

        # Greedy decode for one sentence at a time
        translated_tokens = greedy_decode(model, src, src_padding_mask, max_len=128, 
                                          start_symbol=BOS_IDX, 
                                          end_symbol=EOS_IDX, device).squeeze(0)
        translated_tokens_list = translated_tokens.tolist()

        # Post-process translated tokens to remove special tokens for sacrebleu
        filtered_hyp_tokens = [token_id for token_id in translated_tokens_list if token_id != PAD_IDX]
        hypothesis = tokenizer.decode(filtered_hyp_tokens, skip_special_tokens=True)
        list_of_hypotheses.append(hypothesis)

        # Decode reference target (tgt_label) from the original validation dataset
        ref_tokens_list = tgt_label.squeeze(0).tolist()
        filtered_ref_tokens = [token_id for token_id in ref_tokens_list if token_id != PAD_IDX]
        reference = tokenizer.decode(filtered_ref_tokens, skip_special_tokens=True)
        list_of_references.append([reference]) # sacrebleu expects a list of lists for references


# Calculate BLEU score
if list_of_hypotheses and list_of_references:
    bleu = sacrebleu.corpus_bleu(list_of_hypotheses, list_of_references)
    print(f"BLEU score: {bleu.score:.2f}")
else:
    print("Could not calculate BLEU score: no hypotheses or references generated.")


print("\n--- Training and Validation Loss Curve ---")
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Training Loss')
plt.plot(range(1, EPOCHS + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()


# Assumptions and Simplifications

This reproduction notebook implements a simplified version of the Transformer model from "Attention Is All You Need" due to computational constraints and the goal of a runnable Colab demonstration. Key simplifications include:

1.  **Significantly Reduced Model Size**: The Transformer model uses `d_model=128`, `nhead=4`, `d_hid=256`, and `nlayers=2` for both encoder and decoder. The original paper used much larger models (e.g., `d_model=512`, `nhead=8`, `d_hid=2048`, `nlayers=6`). This reduction is critical for fast training.

2.  **Small Dataset Subset**: Training is performed on only the first 5000 samples of the WMT 2014 English-to-German training data. The original paper used the full WMT 2014 dataset, which contains millions of sentence pairs. A small subset (200-500 samples) is also used for validation and BLEU calculation for speed.

3.  **Simplified Tokenization and Special Tokens**: 
    *   A pre-trained `Helsinki-NLP/opus-mt-en-de` tokenizer from the `transformers` library is used. This tokenizer implicitly handles many aspects of tokenization.
    *   For the Transformer's `BOS_IDX` (start-of-sentence) and `EOS_IDX` (end-of-sentence), we've used `tokenizer.convert_tokens_to_ids('</s>')` which maps to ID `0` for this specific tokenizer. This means our `BOS_IDX` and `EOS_IDX` are effectively the same value (`0`). While the original paper used distinct BOS and EOS tokens, this simplification is made to integrate with the pre-trained tokenizer's special token handling. The `PAD_IDX` is `tokenizer.pad_token_id` (typically `65000`).

4.  **Fixed Learning Rate**: A fixed learning rate of `0.0001` with the Adam optimizer is used. The original paper employed a custom learning rate scheduler with warmup steps, which is crucial for optimal Transformer training but is omitted here for simplicity.

5.  **Limited Training Epochs**: The model is trained for only `2` epochs. The original paper trained for days on multiple GPUs. This short training duration is necessary to complete execution within typical Colab GPU time limits (under 10 minutes).

6.  **Approximated BLEU Evaluation**: BLEU score calculation is performed on a very small subset of the validation data (a few hundred samples) and serves as a quick sanity check rather than a rigorous benchmark. The scores obtained will be significantly lower than those reported in the original paper due to the limited data and model size.

7.  **No Beam Search for Inference**: Translations during evaluation are generated using greedy decoding. The original paper (and most production NMT systems) employs more sophisticated decoding strategies like beam search, which typically yield higher quality translations and better BLEU scores.

These simplifications allow for a complete, end-to-end demonstration of the Transformer architecture and its training loop in PyTorch, while acknowledging the substantial differences from the full-scale research experiment.